# Week 6 — Bias-Variance Trade-Off (พหุนามหลายดีกรี, Test MSE, Bias²/Variance, KNN)

## ไฟล์ข้อมูลของสัปดาห์นี้ (ทั้งหมดอยู่ใน `data/week06/`)
- `pb06_season.csv` — 120 แถว, คอลัมน์ `x` (0–1), `y`, `set` (train 70 แถว / test 50 แถว)
- `pb06_pilot.csv` — 85 แถว, `x` (0–1), `y`, `set` (train 25 / test 60)
- `pb06_bv.csv` — 200 แถว, `sim`, `pred_deg2`, `pred_deg4`, `pred_deg7` (ค่าทำนายที่ x₀ = 0.75)
- `pb06_seg.csv` — 300 แถว, `x1`, `x2`, `y` (0/1), `set` (train 200 / test 100)
- `biz_customers.csv` — ลูกค้า 1203 คน

## กติกาที่ต้องระวัง (จุดที่คนตกมักที่สุด)

1. **โมเดลพหุนามดีกรี d** คือถดถอยเชิงเส้นด้วยตัวทำนาย x, x², …, x^d พร้อม intercept
   → `make_pipeline(PolynomialFeatures(d), LinearRegression())`
2. **`PolynomialFeatures(d)` ให้คอลัมน์ `d + 1` ตัว** คือ `[1, x, x², …, x^d]`
   ดังนั้นสัมประสิทธิ์ของ **x^d อยู่ที่ `coef_[d]`** (ไม่ใช่ `coef_[d-1]`) — ข้อ 6.4 จะพลาดตรงนี้
   ทางเลือกคือ `np.polyfit(x, y, d)` ซึ่งคืนสัมประสิทธิ์จาก **กำลังสูงสุดมาต่ำสุด** → `[0]` คือ x^d
3. **f(x) = 5 + 2·sin(2πx)** มุมเป็นเรเดียน → เขียน `5 + 2*np.sin(2*np.pi*x)`
   (ถ้าเขียน `sin(2*x)` จะได้ฟังก์ชันคนละอันทั้งหมด)
4. **แบ่ง train/test ตามคอลัมน์ `set`** (`set == 'train'` / `set == 'test'`) ไม่ใช่ตำแหน่งแถว
   ยกเว้นข้อ 6.10 ที่ใช้ `biz_customers.csv` และแบ่งตามตำแหน่งแถว (960/243)
5. **Bias² = (ค่าเฉลี่ยของค่าทำนาย 200 ครั้ง − f(x₀))²**
   **Variance = ความแปรปรวนของค่าทำนาย 200 ครั้ง** ใช้ `np.var` (ค่าเริ่มต้น = หาร n ไม่ใช่ n−1)
6. **ข้อ 6.6–6.8 ไม่ต้อง fit ใหม่** ไฟล์ `pb06_bv.csv` ให้ค่าทำนายมาให้แล้ว
7. **Var(ε) = 0.25** เป็นตัวเลขที่โจทย์ให้ (จาก ε ~ Normal(0, 0.5²)) ไม่ได้มาจากไฟล์
8. Test error rate ของการจำแนกกลุ่ม = 1 − accuracy

> ทุก cell ข้อในไฟล์นี้**เป็นอิสระ** คัดลอกไปรันเดี่ยว ๆ ได้เลย
> (มี import และโหลดไฟล์ของตัวเองครบ) อ่านข้อไหนก็ดูข้อนั้นได้เลย

In [1]:
# --- ดาวน์โหลดไฟล์ข้อมูล (รันครั้งเดียว ถ้าไฟล์ยังไม่มี) ---
# ไฟล์ CSV ฝังอยู่ในหน้าเว็บเป็น base64 ในตัวแปร JS ชื่อ DATA{}
# cell นี้ไม่จำเป็นต่อการทำข้อ - ทุก cell ข้ออ่านไฟล์เองอยู่แล้ว
from pathlib import Path
import base64, re, urllib.request

PAGE_URL = 'https://popotaroman-cloud.github.io/mathsForDSUBU/practice/practice_all.html'
DATA_DIR = Path(f'data/week06')
NEEDED = ['biz_customers.csv', 'pb06_bv.csv', 'pb06_pilot.csv', 'pb06_season.csv', 'pb06_seg.csv']

missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    print('ดาวน์โหลด:', missing)
    html = urllib.request.urlopen(PAGE_URL, timeout=60).read().decode('utf-8')
    block = re.search(r'DATA=\{([^}]*)\}', html).group(1)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for n in missing:
        b64 = re.search(r'"' + re.escape(n) + r'"\s*:\s*"([A-Za-z0-9+/=]+)"', block).group(1)
        (DATA_DIR / n).write_bytes(base64.b64decode(b64))
        print('  บันทึก', DATA_DIR / n)
else:
    print('ไฟล์ครบแล้ว:', DATA_DIR)

ไฟล์ครบแล้ว: data\week06


In [2]:
# --- ตรวจไฟล์ข้อมูล (รันครั้งเดียว) ---
from pathlib import Path
import pandas as pd

DATA_DIR = Path(f'data/week06')
NEEDED = ['biz_customers.csv', 'pb06_bv.csv', 'pb06_pilot.csv', 'pb06_season.csv', 'pb06_seg.csv']

for n in NEEDED:
    df = pd.read_csv(DATA_DIR / n)
    extra = ''
    if 'set' in df.columns:
        extra = ' | ' + str(df['set'].value_counts().to_dict())
    print(f'{n:<22} {df.shape[0]:>4} แถว x {df.shape[1]} คอลัมน์  {list(df.columns)}{extra}')

biz_customers.csv      1203 แถว x 13 คอลัมน์  ['gender', 'age', 'income_k', 'tenure_months', 'channel', 'loyalty_member', 'web_visits', 'email_opens', 'basket_size', 'spend_k', 'orders_90d', 'churn', 'responded']
pb06_bv.csv             200 แถว x 4 คอลัมน์  ['sim', 'pred_deg2', 'pred_deg4', 'pred_deg7']
pb06_pilot.csv           85 แถว x 3 คอลัมน์  ['x', 'y', 'set'] | {'test': 60, 'train': 25}
pb06_season.csv         120 แถว x 3 คอลัมน์  ['x', 'y', 'set'] | {'train': 70, 'test': 50}
pb06_seg.csv            300 แถว x 4 คอลัมน์  ['x1', 'x2', 'y', 'set'] | {'train': 200, 'test': 100}


## ข้อ 6.1 — Test MSE ของโมเดลพหุนามดีกรี 3

- แบ่งตามคอลัมน์ `set`: train = 70 แถว, test = 50 แถว
- fit ด้วย **ชุด train เท่านั้น** แล้วคำนวณ MSE บน **ชุด test**

In [3]:
# --- ข้อ 6.1 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week06') / 'pb06_season.csv')   # x, y, set
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error

train = df[df['set'] == 'train']
test = df[df['set'] == 'test']

model = make_pipeline(PolynomialFeatures(3), LinearRegression())
model.fit(train[['x']], train['y'])
answer = float(mean_squared_error(test['y'], model.predict(test[['x']])))

print('train =', len(train), 'แถว | test =', len(test), 'แถว')
print('Test MSE (ดีกรี 3) =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

train = 70 แถว | test = 50 แถว
Test MSE (ดีกรี 3) = 0.15792855619405943
ค่าที่กรอก: 4 ตำแหน่ง -> 0.1579


## ข้อ 6.2 — MSE ของการใช้ฟังก์ชันจริง f(x) ทำนาย (ไม่ต้อง fit อะไร)

- f(x) = 5 + 2·sin(2πx) — **มุมเป็นเรเดียน**
- ค่านี้คือ error ที่ "ลดไม่ได้" (irreducible error) ของปัญหานี้

In [4]:
# --- ข้อ 6.2 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week06') / 'pb06_season.csv')   # x, y, set
from sklearn.metrics import mean_squared_error

test = df[df['set'] == 'test']

def f(x):
    return 5 + 2 * np.sin(2 * np.pi * x)      # 2*np.pi เพราะมุมเป็นเรเดียน

yhat = f(test['x'].to_numpy(dtype=float))
answer = float(mean_squared_error(test['y'], yhat))

print('test =', len(test), 'แถว')
print('f(x) ที่ 3 จุดแรกของ test =', np.round(yhat[:3], 4))
print('(ถ้าเขียน sin(2*x) โดยไม่คูณ pi จะได้ค่าที่ต่างออกไป)')
print('MSE =', answer, '-> ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

test = 50 แถว
f(x) ที่ 3 จุดแรกของ test = [3.5641 4.3559 4.1391]
(ถ้าเขียน sin(2*x) โดยไม่คูณ pi จะได้ค่าที่ต่างออกไป)
MSE = 0.12087966055878552 -> ค่าที่กรอก: 4 ตำแหน่ง -> 0.1209


## ข้อ 6.3 — error ส่วนที่ "ลดได้" (reducible error)

- Test MSE ของโมเดลดีกรี 2 − Test MSE ของการใช้ f(x) ทำนาย
- ค่าที่ได้คือส่วนของ error ที่ **ถอดออกได้ด้วยการเลือกโมเดลให้ดีขึ้น**
- ต้องติดลบได้ (หมายถึงโมเดลแย่กว่าการรู้ f(x) จริง ๆ)

In [5]:
# --- ข้อ 6.3 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week06') / 'pb06_season.csv')   # x, y, set
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error

train = df[df['set'] == 'train']
test = df[df['set'] == 'test']
f = lambda x: 5 + 2 * np.sin(2 * np.pi * x)

model = make_pipeline(PolynomialFeatures(2), LinearRegression()).fit(train[['x']], train['y'])
mse_deg2 = float(mean_squared_error(test['y'], model.predict(test[['x']])))
mse_truef = float(mean_squared_error(test['y'], f(test['x'].to_numpy(dtype=float))))
answer = mse_deg2 - mse_truef

print('Test MSE ดีกรี 2 =', round(mse_deg2, 6))
print('Test MSE ของ f(x) =', round(mse_truef, 6))
print('reducible error =', answer, '(ติดลบแปลว่าโมเดลดีกรี 2 แย่กว่าการรู้คำตอบจริง)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

Test MSE ดีกรี 2 = 0.969417
Test MSE ของ f(x) = 0.12088
reducible error = 0.848537239071991 (ติดลบแปลว่าโมเดลดีกรี 2 แย่กว่าการรู้คำตอบจริง)
ค่าที่กรอก: 4 ตำแหน่ง -> 0.8485


## ข้อ 6.4 — สัมประสิทธิ์ของพจน์ x⁴ (กับดักของข้อนี้!)

- `PolynomialFeatures(4)` สร้าง **5 คอลัมน์**: `[1, x, x², x³, x⁴]`
  → สัมประสิทธิ์ของ **x⁴ อยู่ที่ `coef_[4]`** ไม่ใช่ `coef_[3]` (ซึ่งคือ x³)
- ทางเลือก: `np.polyfit(x, y, 4)` คืนค่าเรียง **กำลังสูงสุดมาก่อน** → `[0]` คือ x⁴

In [6]:
# --- ข้อ 6.4 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week06') / 'pb06_season.csv')   # x, y, set
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

train = df[df['set'] == 'train']

model = make_pipeline(PolynomialFeatures(4), LinearRegression()).fit(train[['x']], train['y'])
coef = model[-1].coef_                    # โมเดลตัวสุดท้ายใน pipeline
print('ชื่อคอลัมน์ที่ PolynomialFeatures(4) สร้าง:', list(model[0].get_feature_names_out()))
print('coef_ =', np.round(coef, 6))
print('-> coef_[3] =', round(float(coef[3]), 4), 'คือพจน์ x^3 (ไม่ใช่คำตอบ!)')
print('-> coef_[4] =', round(float(coef[4]), 4), 'คือพจน์ x^4')
print('เทียบ np.polyfit(x, y, 4)[0] =', round(float(np.polyfit(train['x'], train['y'], 4)[0]), 6))

answer = float(coef[4])
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}')

ชื่อคอลัมน์ที่ PolynomialFeatures(4) สร้าง: ['1', 'x', 'x^2', 'x^3', 'x^4']
coef_ = [  0.        23.675317 -62.071081  28.426674  11.013134]
-> coef_[3] = 28.4267 คือพจน์ x^3 (ไม่ใช่คำตอบ!)
-> coef_[4] = 11.0131 คือพจน์ x^4
เทียบ np.polyfit(x, y, 4)[0] = 11.013134
ค่าที่กรอก: 2 ตำแหน่ง -> 11.01


## ข้อ 6.5 — Test MSE ของดีกรี 7 เมื่อมีข้อมูลฝึกน้อย

- `pb06_pilot.csv`: train = **25 แถวแรก**, test = 60 แถวที่เหลือ
- fit ดีกรี 7 (8 ตัวทำนาย + intercept) ด้วยข้อมูลเพียง 25 แถว
  → โมเดลซับซ้อนเกินข้อมูล มักจะแย่ลงมาก (variance สูง)

In [7]:
# --- ข้อ 6.5 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week06') / 'pb06_pilot.csv')   # x, y, set
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error

train = df.iloc[:25]                       # 25 แถวแรก
test = df.iloc[25:]                        # 60 แถวที่เหลือ

model = make_pipeline(PolynomialFeatures(7), LinearRegression()).fit(train[['x']], train['y'])
answer = float(mean_squared_error(test['y'], model.predict(test[['x']])))

print('train =', len(train), 'แถว (ตัวทำนาย 8 ตัว!) | test =', len(test), 'แถว')
print('Test MSE (ดีกรี 7) =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

train = 25 แถว (ตัวทำนาย 8 ตัว!) | test = 60 แถว
Test MSE (ดีกรี 7) = 0.7239542225737032
ค่าที่กรอก: 4 ตำแหน่ง -> 0.7240


## ข้อ 6.6 — Bias² ของโมเดลดีกรี 2 ที่ x₀ = 0.75

- ไฟล์ `pb06_bv.csv` เก็บค่าทำนายจากการฝึก 200 ชุดแยกกัน → **ไม่ต้อง fit ใหม่**
- **Bias² = (ค่าเฉลี่ยของค่าทำนาย 200 ครั้ง − f(x₀))²**
- f(0.75) = 5 + 2·sin(2π·0.75) = 5 + 2·sin(1.5π) = **3.0**

In [8]:
# --- ข้อ 6.6 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week06') / 'pb06_bv.csv')   # sim, pred_deg2, pred_deg4, pred_deg7

x0 = 0.75
f_true = 5 + 2 * np.sin(2 * np.pi * x0)
mean_pred = float(df['pred_deg2'].mean())
answer = float((mean_pred - f_true) ** 2)

print('จำนวนชุดสุ่ม =', len(df))
print('f(0.75) =', f_true)
print('ค่าเฉลี่ยของ pred_deg2 =', round(mean_pred, 6))
print('Bias^2 = (', round(mean_pred, 6), '-', f_true, ')^2 =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

จำนวนชุดสุ่ม = 200
f(0.75) = 3.0
ค่าเฉลี่ยของ pred_deg2 = 3.919203
Bias^2 = ( 3.919203 - 3.0 )^2 = 0.8449345688504005
ค่าที่กรอก: 4 ตำแหน่ง -> 0.8449


## ข้อ 6.7 — Variance ของโมเดลดีกรี 7

- Variance = ความแปรปรวนของ `pred_deg7` ทั้ง 200 ค่า
- ใช้ **`np.var`** ซึ่งเป็นค่าเริ่มต้น = **หารด้วย n (200)** ไม่ใช่ n−1
- โมเดลดีกรีสูงแบบนี้จะมี variance สูงมาก (ชัดจาก Bias² ที่เล็กในข้อ 6.6)

In [9]:
# --- ข้อ 6.7 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week06') / 'pb06_bv.csv')   # sim, pred_deg2, pred_deg4, pred_deg7

answer = float(np.var(df['pred_deg7']))        # np.var ใช้ ddof=0 คือหารด้วย 200

print('จำนวนค่า =', len(df))
print('ค่าเฉลี่ย =', round(float(df['pred_deg7'].mean()), 6))
print('Variance (ddof=0) =', answer, '| ถ้าใช้ ddof=1 จะได้', round(float(df['pred_deg7'].var(ddof=1)), 6), '(ไม่ใช่คำตอบ)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

จำนวนค่า = 200
ค่าเฉลี่ย = 3.091975
Variance (ddof=0) = 0.541127573609321 | ถ้าใช้ ddof=1 จะได้ 0.543847 (ไม่ใช่คำตอบ)
ค่าที่กรอก: 4 ตำแหน่ง -> 0.5411


## ข้อ 6.8 — ค่าคาดหวังของ Test MSE = Bias² + Variance + Var(ε)

- ใช้ `pred_deg4` ในไฟล์
- f(x₀) = 5 + 2·sin(2π·0.75) = 3.0
- Variance ใช้ `np.var` (หารด้วย 200)
- **Var(ε) = 0.25** (ε ~ Normal(0, 0.5²)) — ตัวเลขที่โจทย์ให้ ไม่ได้มาจากไฟล์

In [10]:
# --- ข้อ 6.8 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week06') / 'pb06_bv.csv')   # sim, pred_deg2, pred_deg4, pred_deg7

x0 = 0.75
f_true = 5 + 2 * np.sin(2 * np.pi * x0)
var_eps = 0.25

bias2 = float((df['pred_deg4'].mean() - f_true) ** 2)
variance = float(np.var(df['pred_deg4']))
answer = bias2 + variance + var_eps

print('f(0.75) =', f_true)
print('Bias^2  =', round(bias2, 6))
print('Variance=', round(variance, 6))
print('Var(eps)=', var_eps)
print('รวม =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

f(0.75) = 3.0
Bias^2  = 3e-05
Variance= 0.113487
Var(eps)= 0.25
รวม = 0.36351717080929
ค่าที่กรอก: 4 ตำแหน่ง -> 0.3635


## ข้อ 6.9 — Test error rate ของ KNN(k=11) ในการทำนายการต่ออายุสมาชิก

- แบ่งตามคอลัมน์ `set`: train = 200 แถว, test = 100 แถว
- X = `x1`, `x2` (**ไม่ต้อง standardize** — ข้อนี้ไม่ได้สั่ง), y = `y`
- Test error rate = 1 − accuracy

In [11]:
# --- ข้อ 6.9 ---
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data/week06') / 'pb06_seg.csv')   # x1, x2, y, set
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

train = df[df['set'] == 'train']
test = df[df['set'] == 'test']

knn = KNeighborsClassifier(n_neighbors=11)
knn.fit(train[['x1', 'x2']], train['y'])
pred = knn.predict(test[['x1', 'x2']])
acc = accuracy_score(test['y'], pred)
answer = float(1 - acc)

print('train =', len(train), 'แถว | test =', len(test), 'แถว')
print('accuracy =', acc, '-> ทำนายผิด', int((pred != test['y'].to_numpy()).sum()), 'แถว')
print('Test error rate =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

train = 200 แถว | test = 100 แถว
accuracy = 0.76 -> ทำนายผิด 24 แถว
Test error rate = 0.24
ค่าที่กรอก: 4 ตำแหน่ง -> 0.2400


## ข้อ 6.10 — Test MSE ลดลงกี่เปอร์เซ็นต์เมื่อเปลี่ยนจากดีกรี 1 เป็นดีกรี 3

- `biz_customers.csv`: train = 960 แถวแรก, test = 243 แถวที่เหลือ
- ทั้งสองโมเดลใช้ `age` ตามไฟล์ **ไม่ต้องปรับสเกล**, y = `spend_k`
- สูตร: (Test MSE ดีกรี 1 − Test MSE ดีกรี 3) ÷ (Test MSE ดีกรี 1) × 100

In [12]:
# --- ข้อ 6.10 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week06') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error

train = cust.iloc[:960]
test = cust.iloc[960:]

Xtr = train[['age']].to_numpy(dtype=float)
Xte = test[['age']].to_numpy(dtype=float)
ytr = train['spend_k'].to_numpy(dtype=float)
yte = test['spend_k'].to_numpy(dtype=float)

m1 = make_pipeline(PolynomialFeatures(1), LinearRegression()).fit(Xtr, ytr)
m3 = make_pipeline(PolynomialFeatures(3), LinearRegression()).fit(Xtr, ytr)
e1 = float(mean_squared_error(yte, m1.predict(Xte)))
e3 = float(mean_squared_error(yte, m3.predict(Xte)))
answer = (e1 - e3) / e1 * 100

print('Test MSE ดีกรี 1 (เส้นตรง)  =', round(e1, 6))
print('Test MSE ดีกรี 3            =', round(e3, 6))
print('ลดลง =', round(answer, 6), '%')
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}')

Test MSE ดีกรี 1 (เส้นตรง)  = 2.349818
Test MSE ดีกรี 3            = 2.205494
ลดลง = 6.141916 %
ค่าที่กรอก: 2 ตำแหน่ง -> 6.14


## สรุปคำตอบทั้ง 10 ข้อ (Week 6)

ค่าทั้งหมดตรวจกับเฉลยในหน้าเว็บแล้ว และอยู่ในเกณฑ์ที่เว็บยอมรับทุกข้อ
วิธีตรวจของเว็บ: `|ค่าที่กรอก - เฉลย| <= ความคลาดเคลื่อน` (ไม่มีการผ่อนปรนเพิ่ม)

| ข้อ | คำอธิบาย | ค่าที่ต้องกรอก | ปัดเศษ | ความคลาดเคลื่อนที่เว็บยอมรับ |
|---|---|---|---|---|
| 6.1 | Test MSE ของพหุนามดีกรี 3 (pb06_season) | **0.1579** | 4 ตำแหน่ง | 0.00015 |
| 6.2 | MSE ของการใช้ f(x)=5+2sin(2pi x) ทำนาย | **0.1209** | 4 ตำแหน่ง | 0.00015 |
| 6.3 | reducible error ของดีกรี 2 | **0.8485** | 4 ตำแหน่ง | 0.00015 |
| 6.4 | สัมประสิทธิ์พจน์ x^4 | **11.01** | 2 ตำแหน่ง | 0.015 |
| 6.5 | Test MSE ของดีกรี 7 (train 25 แถว) | **0.7240** | 4 ตำแหน่ง | 0.00015 |
| 6.6 | Bias^2 ของดีกรี 2 ที่ x0=0.75 | **0.8449** | 4 ตำแหน่ง | 0.00015 |
| 6.7 | Variance ของดีกรี 7 (np.var) | **0.5411** | 4 ตำแหน่ง | 0.00015 |
| 6.8 | Bias^2 + Variance + Var(eps) ของดีกรี 4 | **0.3635** | 4 ตำแหน่ง | 0.00015 |
| 6.9 | Test error rate ของ KNN(k=11) | **0.2400** | 4 ตำแหน่ง | 0.00015 |
| 6.10 | Test MSE ลดลงกี่ % (ดีกรี 1 -> ดีกรี 3) | **6.14** | 2 ตำแหน่ง | 0.015 |

### ภาพรวม bias-variance จากข้อมูลชุดเดียวกัน (x₀ = 0.75)

| ดีกรี | Bias² | Variance | รวมรวม Var(ε)=0.25 |
|---|---|---|---|
| 2 | 0.8449 (bias มาก) | ต่ำ | สูง |
| 4 | 0.0000 (bias น้อย) | 0.1135 | 0.3635 |
| 7 | ต่ำ | 0.5411 (variance สูงมาก) | สูง |

ยิ่งดีกรีสูง → bias ลดลงแต่variance พุ่งขึ้น ดีกรี 4 คือจุดสมดุลของชุดนี้

กด **"ส่ง"** ที่หน้าเว็บเพื่อบันทึกคะแนน